# Beam search vs greedy: BLEU-4 as a function of beam size

The training and eval notebooks all decode **greedily** (argmax at every step). This notebook loads one trained checkpoint and re-decodes the same images with **beam search** at several beam widths, then plots **BLEU-4 vs beam size** (beam size 1 == greedy).

Beam search is a *decode-time* change only — no retraining. Hugging Face's `generate(num_beams=...)` can't be used here because our decoders manually feed `encoder_hidden_states` for cross-attention, so beam search is implemented directly for both the **GPT-2** and **GRU** decoders. The checkpoint stores its own `config`, so the right model is rebuilt automatically. Built for Colab (also runs locally).

## 1. Install dependencies and imports

In [ ]:
# Install once if needed
!pip -q install transformers pycocotools nltk

import os, json, random, time, copy, math, textwrap, shutil
from dataclasses import dataclass, asdict, fields
from typing import Optional
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision import models

from transformers import (
    AutoConfig, AutoModel, AutoModelForCausalLM, AutoTokenizer,
    AutoImageProcessor, CLIPVisionModel,
)

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Using device:", device)

## 2. Settings & mount Drive

`MODEL_PATH` points at the checkpoint to evaluate. `BEAM_SIZES` is the sweep (1 = greedy). Beam search runs **per image**, so cost scales with beam size — by default it evaluates the clean held-out validation images (`EVAL_SPLIT="val"`, ~10%); raise `NUM_EVAL_IMAGES` or switch to `"all"` for more (slower).

In [ ]:
# Portable paths: works on Colab (mounts Drive) AND a local Jupyter notebook.
try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content"
    ON_COLAB = True
except ImportError:
    BASE_DIR = os.path.abspath(".")
    ON_COLAB = False

DATA_DIR = os.path.join(BASE_DIR, "data", "coco")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs_beam")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- The checkpoint to evaluate (edit this) ---------------------------------
MODEL_PATH = "/content/drive/MyDrive/image_captioning_finetune/vit_gpt2_best.pt"

# ---- What to evaluate on -----------------------------------------------------
COCO_SPLIT = "val"        # which COCO image set is on disk
EVAL_SPLIT = "val"        # "val" = clean held-out 10% (recommended for a metric); "all" = every image
NUM_EVAL_IMAGES = None    # e.g. 300 for a quick pass; None = all in EVAL_SPLIT

# ---- Beam search sweep -------------------------------------------------------
BEAM_SIZES = [1, 2, 3, 5]   # 1 == greedy
LENGTH_PENALTY = 1.0        # >1 favours longer captions, <1 favours shorter (length-normalised score)
MAX_GEN_LEN = 40
MAX_TEXT_LEN = 40

print("Model path:", MODEL_PATH)
print("Output dir:", OUTPUT_DIR)
print("Beam sizes:", BEAM_SIZES, "| eval split:", EVAL_SPLIT, "| cap:", NUM_EVAL_IMAGES)

## 3. Download MS-COCO data

Pure-Python download/extract; skips the work if the data is already present.

In [ ]:
import urllib.request, zipfile

ANNOTATIONS_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
IMAGES_URL = ("http://images.cocodataset.org/zips/val2017.zip" if COCO_SPLIT == "val"
              else "http://images.cocodataset.org/zips/train2017.zip")

ann_dir = os.path.join(DATA_DIR, "annotations")
img_dir = os.path.join(DATA_DIR, f"{COCO_SPLIT}2017")

def _download_and_extract(url, zip_path, extract_to):
    def _progress(block_num, block_size, total_size):
        if total_size > 0:
            pct = min(100, block_num * block_size * 100 / total_size)
            print(f"\r  downloading... {pct:5.1f}%", end="")
    print("Downloading", url)
    urllib.request.urlretrieve(url, zip_path, _progress)
    print("\n  extracting...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    os.remove(zip_path)
    print("  done.")

if not os.path.exists(ann_dir):
    _download_and_extract(ANNOTATIONS_URL, os.path.join(DATA_DIR, "annotations.zip"), DATA_DIR)
else:
    print("Annotations already extracted.")

if not os.path.exists(img_dir):
    _download_and_extract(IMAGES_URL, os.path.join(DATA_DIR, f"{COCO_SPLIT}2017.zip"), DATA_DIR)
else:
    print(f"{COCO_SPLIT}2017 images already extracted.")

## 4. Load captions and reproduce the train/val split

Same seed-42 90/10 split-by-image-id as training, so `EVAL_SPLIT="val"` lines up with the model's held-out set.

In [ ]:
annotations_file = os.path.join(DATA_DIR, "annotations", f"captions_{COCO_SPLIT}2017.json")
with open(annotations_file, "r") as f:
    coco_data = json.load(f)

img_id_to_filename = {img["id"]: img["file_name"] for img in coco_data["images"]}
img_id_to_captions = defaultdict(list)
for ann in coco_data["annotations"]:
    img_id_to_captions[ann["image_id"]].append(ann["caption"])

def make_image_id_split(image_ids, train_fraction=0.90, seed=42):
    image_ids = list(image_ids)
    rng = random.Random(seed)
    rng.shuffle(image_ids)
    split_idx = int(train_fraction * len(image_ids))
    return set(image_ids[:split_idx]), set(image_ids[split_idx:])

train_img_ids, val_img_ids = make_image_id_split(img_id_to_filename.keys(), 0.90, SEED)
train_annotations = [a for a in coco_data["annotations"] if a["image_id"] in train_img_ids]

print("Total images:", len(img_id_to_filename))
print("Train images:", len(train_img_ids), " Val images:", len(val_img_ids))

## 5. Unified encoder/decoder framework

Same building blocks the training notebooks used, so a checkpoint's saved `config` rebuilds the identical model.

In [ ]:
# ---- Word-level vocabulary (for GRU checkpoints) ----------------------------
class Vocabulary:
    def __init__(self, freq_threshold=5):
        self.freq_threshold = freq_threshold
        self.word2idx = {"<pad>": 0, "<start>": 1, "<end>": 2, "<unk>": 3}
        self.idx2word = {0: "<pad>", 1: "<start>", 2: "<end>", 3: "<unk>"}
        self.word_count = Counter()

    def build(self, captions):
        for cap in captions:
            self.word_count.update(word_tokenize(cap.lower()))
        idx = 4
        for w, c in self.word_count.items():
            if c >= self.freq_threshold:
                self.word2idx[w] = idx; self.idx2word[idx] = w; idx += 1

    def decode(self, ids):
        words = []
        for i in ids:
            w = self.idx2word.get(int(i), "<unk>")
            if w == "<end>": break
            if w not in ("<start>", "<pad>"): words.append(w)
        return " ".join(words)

    def __len__(self):
        return len(self.word2idx)

rnn_vocab = Vocabulary(freq_threshold=5)
rnn_vocab.build([a["caption"] for a in train_annotations])
print("GRU word-level vocab size:", len(rnn_vocab))

# ---- Image preprocessing ----------------------------------------------------
resnet_transform = T.Compose([
    T.Resize((256, 256)), T.CenterCrop(224), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

def preprocess_images(images, encoder_kind, image_processor):
    if encoder_kind == "cnn":
        return torch.stack([resnet_transform(im) for im in images], 0)
    return image_processor(list(images), return_tensors="pt").pixel_values

# ---- Encoder: image -> sequence of feature vectors (B, S, D_enc) ------------
class ImageEncoder(nn.Module):
    def __init__(self, kind, name):
        super().__init__()
        self.kind = kind
        if kind == "cnn":
            resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            self.backbone = nn.Sequential(*list(resnet.children())[:-2])
            self.feat_dim = 2048
        elif kind == "clip":
            self.backbone = CLIPVisionModel.from_pretrained(name)
            self.feat_dim = self.backbone.config.hidden_size
        else:  # vit
            self.backbone = AutoModel.from_pretrained(name)
            self.feat_dim = self.backbone.config.hidden_size

    def forward(self, images):
        if self.kind == "cnn":
            f = self.backbone(images)
            B, C, H, W = f.shape
            return f.view(B, C, H * W).permute(0, 2, 1)
        out = self.backbone(pixel_values=images)
        return out.last_hidden_state

# ---- Decoder A: word-level GRU ----------------------------------------------
class GRUDecoder(nn.Module):
    def __init__(self, feat_dim, vocab, embed_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.vocab = vocab
        self.pad_id = vocab.word2idx["<pad>"]
        self.end_id = vocab.word2idx["<end>"]
        self.img_proj = nn.Linear(feat_dim, embed_size)
        self.bn = nn.BatchNorm1d(embed_size)
        self.embed = nn.Embedding(len(vocab), embed_size)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, len(vocab))

    def _img_token(self, enc_seq):
        pooled = enc_seq.mean(dim=1)
        return self.bn(self.img_proj(pooled))

    def decode(self, ids):
        return self.vocab.decode(ids)

# ---- Decoder B: GPT-2 with cross-attention ----------------------------------
class GPT2Decoder(nn.Module):
    def __init__(self, feat_dim, tokenizer, dropout=0.1, freeze_base=False):
        super().__init__()
        cfg = AutoConfig.from_pretrained("gpt2")
        cfg.is_decoder = True
        cfg.add_cross_attention = True
        cfg.resid_pdrop = dropout
        cfg.embd_pdrop = dropout
        cfg.attn_pdrop = dropout
        self.gpt2 = AutoModelForCausalLM.from_pretrained("gpt2", config=cfg)
        self.gpt2.resize_token_embeddings(len(tokenizer))
        self.enc_proj = nn.Linear(feat_dim, cfg.n_embd)
        self.tokenizer = tokenizer
        self.bos_id = tokenizer.bos_token_id
        self.eos_id = tokenizer.eos_token_id
        self.pad_id = tokenizer.pad_token_id

    def decode(self, ids):
        return self.tokenizer.decode(ids, skip_special_tokens=True).strip()

# ---- Full model -------------------------------------------------------------
class CaptioningModel(nn.Module):
    def __init__(self, encoder, decoder, freeze_encoder=True):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.freeze_encoder = freeze_encoder

    @torch.no_grad()
    def encode(self, images):
        return self.encoder(images)

    def decode(self, ids):
        return self.decoder.decode(ids)

# ---- Config + builder -------------------------------------------------------
@dataclass
class ExperimentConfig:
    name: str
    encoder_kind: str
    encoder_name: str
    decoder: str
    learning_rate: float
    weight_decay: float = 0.0
    dropout: float = 0.1
    freeze_gpt2_base: bool = False
    embed_size: int = 256
    hidden_size: int = 512
    num_layers: int = 2
    freq_threshold: int = 5
    batch_size: int = 16
    epochs: int = 15
    max_train_batches: Optional[int] = 300

def build_model(config):
    encoder = ImageEncoder(config.encoder_kind, config.encoder_name)
    feat_dim = encoder.feat_dim
    if config.decoder == "gru":
        decoder = GRUDecoder(feat_dim, rnn_vocab, config.embed_size,
                             config.hidden_size, config.num_layers, config.dropout)
    elif config.decoder == "gpt2":
        tok = AutoTokenizer.from_pretrained("gpt2")
        if tok.pad_token is None:
            tok.add_special_tokens({"pad_token": "<|pad|>"})
        decoder = GPT2Decoder(feat_dim, tok, dropout=config.dropout,
                              freeze_base=config.freeze_gpt2_base)
    else:
        raise ValueError(f"Unknown decoder: {config.decoder}")
    model = CaptioningModel(encoder, decoder, freeze_encoder=True)
    image_processor = (None if config.encoder_kind == "cnn"
                       else AutoImageProcessor.from_pretrained(config.encoder_name))
    return {"model": model, "encoder_kind": config.encoder_kind,
            "decoder_kind": config.decoder, "image_processor": image_processor}

print("Framework ready.")

## 6. Load the checkpoint

Rebuilds the exact architecture from the `config` stored in the checkpoint, then loads the trained weights.

In [ ]:
ckpt = torch.load(MODEL_PATH, map_location=device)
# Some checkpoints (e.g. the ViT-only notebook) don't store every field this
# notebook's ExperimentConfig expects. Infer/fill the missing ones.
saved_cfg = dict(ckpt["config"])
if "encoder_kind" not in saved_cfg:
    _nm = str(saved_cfg.get("encoder_name", "")).lower()
    if "clip" in _nm:
        saved_cfg["encoder_kind"] = "clip"
    elif "resnet" in _nm or _nm == "cnn":
        saved_cfg["encoder_kind"] = "cnn"
    else:
        saved_cfg["encoder_kind"] = "vit"
saved_cfg.setdefault("name", os.path.splitext(os.path.basename(MODEL_PATH))[0])
saved_cfg.setdefault("decoder", "gpt2")

valid = {f.name for f in fields(ExperimentConfig)}
cfg_dict = {k: v for k, v in saved_cfg.items() if k in valid}
config = ExperimentConfig(**cfg_dict)
MODEL_NAME = config.name

bundle = build_model(config)
model = bundle["model"].to(device)
missing, unexpected = model.load_state_dict(ckpt["state_dict"], strict=False)
model.eval()

print("Loaded:", MODEL_NAME)
print(f"  encoder = {config.encoder_kind} ({config.encoder_name})")
print(f"  decoder = {config.decoder}")
print(f"  reported val BLEU-4 at save time = {ckpt.get('bleu4')}  (greedy, epoch {ckpt.get('epoch')})")
if missing:    print("  [warn] missing keys:", len(missing))
if unexpected: print("  [warn] unexpected keys:", len(unexpected))

## 7. Beam search decoders

Both implementations keep `beam_size` active hypotheses ranked by cumulative log-probability. A hypothesis that emits the end token is moved to a *finished* pool and scored with **length normalisation** (`score / length**LENGTH_PENALTY`) so beam search isn't biased toward very short captions. At the end the best finished hypothesis wins (falling back to the best active one). **`beam_size=1` reduces exactly to greedy**, so it's a clean baseline using the same code path.

- **GPT-2**: re-runs the decoder over the growing token sequence each step (no KV-cache), feeding `encoder_hidden_states` every time — simplest correct way to handle the cross-attention beam expansion.
- **GRU**: carries a per-beam hidden state, batched along the beam dimension and reordered as beams are selected.

In [ ]:
@torch.no_grad()
def beam_search_gpt2(decoder, enc_seq_1, max_len, beam_size, length_penalty=1.0):
    # enc_seq_1: (1, S, D_enc). Returns a list of GPT-2 token ids (no bos/eos).
    dev = enc_seq_1.device
    enc_hidden = decoder.enc_proj(enc_seq_1)          # (1, S, n_embd)
    bos, eos = decoder.bos_id, decoder.eos_id
    beams = [([bos], 0.0)]                              # (tokens incl. bos, cum logprob)
    finished = []
    for _ in range(max_len):
        if not beams:
            break
        input_ids = torch.tensor([b[0] for b in beams], device=dev)   # (nb, L) equal length
        nb = input_ids.size(0)
        enc_b = enc_hidden.expand(nb, -1, -1).contiguous()
        out = decoder.gpt2(input_ids=input_ids, encoder_hidden_states=enc_b)
        logprobs = F.log_softmax(out.logits[:, -1, :], dim=-1)        # (nb, V)
        V = logprobs.size(-1)
        total = torch.tensor([b[1] for b in beams], device=dev).unsqueeze(1) + logprobs
        cand = min(2 * beam_size, total.numel())
        top_scores, top_flat = total.view(-1).topk(cand)
        new_beams = []
        for s, fi in zip(top_scores.tolist(), top_flat.tolist()):
            b, t = divmod(int(fi), V)
            seq = beams[b][0] + [t]
            if t == eos:
                norm = s / (max(len(seq) - 1, 1) ** length_penalty)   # exclude bos
                finished.append((seq, norm))
            else:
                new_beams.append((seq, s))
            if len(new_beams) == beam_size:
                break
        beams = new_beams
        if len(finished) >= beam_size:
            break
    if finished:
        best = max(finished, key=lambda x: x[1])[0]
    else:
        best = max(beams, key=lambda b: b[1] / (max(len(b[0]) - 1, 1) ** length_penalty))[0]
    return [t for t in best if t not in (bos, eos)]


@torch.no_grad()
def beam_search_gru(decoder, enc_seq_1, max_len, beam_size, length_penalty=1.0):
    # enc_seq_1: (1, S, D_enc). Returns a list of word ids (no start/end).
    dev = enc_seq_1.device
    end = decoder.end_id
    feat = decoder._img_token(enc_seq_1)               # (1, E)
    out, states = decoder.rnn(feat.unsqueeze(1))        # out (1,1,H), states (L,1,H)
    logprobs = F.log_softmax(decoder.linear(out.squeeze(1)), dim=-1)[0]  # (V,)
    topv, topi = logprobs.topk(beam_size)
    seqs = [[int(t)] for t in topi.tolist()]
    scores = topv.clone()                               # (k,)
    states = states.repeat(1, beam_size, 1)             # (L, k, H)
    finished = []
    for _ in range(max_len - 1):
        if not seqs:
            break
        last = torch.tensor([s[-1] for s in seqs], device=dev)
        emb = decoder.embed(last).unsqueeze(1)          # (nb,1,E)
        out, states = decoder.rnn(emb, states)
        lp = F.log_softmax(decoder.linear(out.squeeze(1)), dim=-1)      # (nb, V)
        V = lp.size(-1)
        total = scores.unsqueeze(1) + lp                # (nb, V)
        cand = min(2 * beam_size, total.numel())
        top_scores, top_flat = total.view(-1).topk(cand)
        new_seqs, new_scores, parent = [], [], []
        for s, fi in zip(top_scores.tolist(), top_flat.tolist()):
            b, t = divmod(int(fi), V)
            if t == end:
                norm = s / (max(len(seqs[b]), 1) ** length_penalty)
                finished.append((seqs[b], norm))
            else:
                new_seqs.append(seqs[b] + [t]); new_scores.append(s); parent.append(b)
            if len(new_seqs) == beam_size:
                break
        if not new_seqs:
            break
        seqs = new_seqs
        scores = torch.tensor(new_scores, device=dev)
        states = states[:, torch.tensor(parent, device=dev), :].contiguous()
        if len(finished) >= beam_size:
            break
    if finished:
        return max(finished, key=lambda x: x[1])[0]
    return max(zip(seqs, scores.tolist()),
               key=lambda z: z[1] / (max(len(z[0]), 1) ** length_penalty))[0]


@torch.no_grad()
def caption_beam(model, pixel_values_1, max_len, beam_size, length_penalty=1.0):
    enc_seq = model.encode(pixel_values_1)              # (1, S, D)
    dec = model.decoder
    if isinstance(dec, GPT2Decoder):
        ids = beam_search_gpt2(dec, enc_seq, max_len, beam_size, length_penalty)
    else:
        ids = beam_search_gru(dec, enc_seq, max_len, beam_size, length_penalty)
    return model.decode(ids)

print("Beam search ready. (beam_size=1 == greedy)")

## 8. Sweep beam size and score BLEU-4

For each beam width, decode every eval image and compute **corpus BLEU-4** and **mean per-image BLEU-4** (smoothing `method1`, against all human references — the training metric). Also tracks average caption length, which usually drifts as beam size grows.

In [ ]:
if EVAL_SPLIT == "val":
    eval_ids = [i for i in img_id_to_filename if i in val_img_ids]
else:
    eval_ids = list(img_id_to_filename.keys())
eval_ids = sorted(eval_ids)
if NUM_EVAL_IMAGES is not None:
    eval_ids = eval_ids[:NUM_EVAL_IMAGES]
print("Comparing on", len(eval_ids), "images | beam sizes", BEAM_SIZES)

kind = bundle["encoder_kind"]
image_processor = bundle["image_processor"]
smoothing = SmoothingFunction().method1
ref_tok = {iid: [word_tokenize(c.lower()) for c in img_id_to_captions[iid]] for iid in eval_ids}

def load_pixel(iid):
    img = Image.open(os.path.join(img_dir, img_id_to_filename[iid])).convert("RGB")
    return preprocess_images([img], kind, image_processor).to(device)

results = []
per_beam_caps = {}
for bs in BEAM_SIZES:
    hyps, refs, caps = [], [], {}
    t0 = time.time()
    for n, iid in enumerate(eval_ids):
        cap = caption_beam(model, load_pixel(iid), MAX_GEN_LEN, bs, LENGTH_PENALTY)
        caps[iid] = cap
        hyps.append(word_tokenize(cap.lower()))
        refs.append(ref_tok[iid])
        if (n + 1) % 50 == 0:
            print(f"\r  beam {bs}: {n+1}/{len(eval_ids)} ({time.time()-t0:.0f}s)", end="")
    print()
    corpus = corpus_bleu(refs, hyps, smoothing_function=smoothing)
    mean = float(np.mean([sentence_bleu(r, h, weights=(0.25, 0.25, 0.25, 0.25),
                                        smoothing_function=smoothing) if h else 0.0
                          for r, h in zip(refs, hyps)]))
    avg_len = float(np.mean([len(h) for h in hyps]))
    results.append({"beam_size": bs, "corpus_bleu4": corpus, "mean_bleu4": mean,
                    "avg_caption_len": avg_len, "seconds": round(time.time() - t0, 1)})
    per_beam_caps[bs] = caps
    print(f"  beam {bs}: corpus BLEU-4 = {corpus:.4f} | mean = {mean:.4f} | "
          f"avg len = {avg_len:.1f} | {time.time()-t0:.0f}s")

res_df = pd.DataFrame(results)
res_csv = os.path.join(OUTPUT_DIR, f"beam_size_comparison_{MODEL_NAME}.csv")
res_df.to_csv(res_csv, index=False)
print("\nsaved:", res_csv)
res_df

## 9. Plot — BLEU-4 vs beam size

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(res_df["beam_size"], res_df["corpus_bleu4"], marker="o", linewidth=2, label="corpus BLEU-4")
ax.plot(res_df["beam_size"], res_df["mean_bleu4"], marker="s", linewidth=2, label="mean per-image BLEU-4")
ax.set_xlabel("beam size  (1 = greedy)")
ax.set_ylabel("BLEU-4")
ax.set_title(f"BLEU-4 vs beam size  -  {MODEL_NAME}")
ax.set_xticks(BEAM_SIZES)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
beam_png = os.path.join(OUTPUT_DIR, f"beam_size_bleu_{MODEL_NAME}.png")
fig.savefig(beam_png, dpi=150, bbox_inches="tight")
plt.show()
print("saved:", beam_png)

best_row = res_df.loc[res_df["corpus_bleu4"].idxmax()]
print(f"Best corpus BLEU-4 at beam size {int(best_row['beam_size'])} "
      f"({best_row['corpus_bleu4']:.4f}) vs greedy "
      f"({res_df.loc[res_df['beam_size']==1,'corpus_bleu4'].iloc[0]:.4f}).")

## 10. Qualitative — how a caption changes with beam size

A few images with the caption produced at each beam width, so you can see beam search fixing greedy artifacts (e.g. repetition).

In [ ]:
sample_ids = eval_ids[:4]
fig, axes = plt.subplots(1, len(sample_ids), figsize=(len(sample_ids) * 4.6, 5.5))
axes = np.array(axes).reshape(-1)
for ax, iid in zip(axes, sample_ids):
    img = Image.open(os.path.join(img_dir, img_id_to_filename[iid])).convert("RGB")
    ax.imshow(img); ax.axis("off")
    lines = [f"b{bs}: {per_beam_caps[bs][iid]}" for bs in BEAM_SIZES]
    ax.set_title("\n".join(textwrap.fill(l, 36) for l in lines), fontsize=7)
fig.suptitle(f"Caption vs beam size  -  {MODEL_NAME}", fontsize=13)
fig.tight_layout()
ex_png = os.path.join(OUTPUT_DIR, f"beam_examples_{MODEL_NAME}.png")
fig.savefig(ex_png, dpi=150, bbox_inches="tight")
plt.show()
print("saved:", ex_png)

## 10b. Test the "greedy causes repetition" hypothesis

Greedy decoding can fall into likelihood loops (e.g. *"a cat next to a cat"*). Beam search explores alternatives, so **if greedy is the cause, repetition should drop as beam size grows.** Using the captions already generated for every beam size (`per_beam_caps`), this measures three repetition signals per beam width:

- **% captions with a repeated content word** (captures *"a cat ... a cat"*),
- **% captions with a repeated bigram**,
- **distinct-2** = unique bigrams / total bigrams (higher = less repetitive).

A downward trend in the first two (and upward in distinct-2) supports the hypothesis.

In [ ]:
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
STOP = set(stopwords.words("english"))

def repetition_stats(caption):
    toks = word_tokenize(caption.lower())
    content = [t for t in toks if t.isalpha() and len(t) > 2 and t not in STOP]
    rep_word = any(c > 1 for c in Counter(content).values())     # repeated content word
    bigrams = list(zip(toks, toks[1:]))
    rep_bigram = any(c > 1 for c in Counter(bigrams).values())   # repeated bigram
    distinct2 = (len(set(bigrams)) / len(bigrams)) if bigrams else 1.0
    return rep_word, rep_bigram, distinct2

rep_rows = []
for bs in BEAM_SIZES:
    flags = [repetition_stats(c) for c in per_beam_caps[bs].values()]
    rw = 100 * np.mean([f[0] for f in flags])
    rb = 100 * np.mean([f[1] for f in flags])
    d2 = float(np.mean([f[2] for f in flags]))
    rep_rows.append({"beam_size": bs, "pct_repeat_content_word": rw,
                     "pct_repeat_bigram": rb, "mean_distinct2": d2})
    print(f"beam {bs}: repeat-content-word {rw:5.1f}% | repeat-bigram {rb:5.1f}% | distinct-2 {d2:.3f}")

rep_df = pd.DataFrame(rep_rows)
rep_csv = os.path.join(OUTPUT_DIR, f"repetition_vs_beam_{MODEL_NAME}.csv")
rep_df.to_csv(rep_csv, index=False)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(rep_df["beam_size"], rep_df["pct_repeat_content_word"], marker="o", linewidth=2,
        label="% captions w/ repeated content word")
ax.plot(rep_df["beam_size"], rep_df["pct_repeat_bigram"], marker="s", linewidth=2,
        label="% captions w/ repeated bigram")
ax.set_xlabel("beam size  (1 = greedy)")
ax.set_ylabel("% of captions")
ax.set_title(f"Repetition vs beam size  -  {MODEL_NAME}")
ax.set_xticks(BEAM_SIZES); ax.grid(True, alpha=0.3); ax.legend()
fig.tight_layout()
rep_png = os.path.join(OUTPUT_DIR, f"repetition_vs_beam_{MODEL_NAME}.png")
fig.savefig(rep_png, dpi=150, bbox_inches="tight"); plt.show()

# Side-by-side: greedy captions that repeat, and what the widest beam produces instead.
print("\nGreedy (beam 1) captions with a repeated content word -> widest-beam version:")
shown = 0
big = BEAM_SIZES[-1]
for iid, cap in per_beam_caps[1].items():
    toks = [t for t in word_tokenize(cap.lower()) if t.isalpha() and len(t) > 2 and t not in STOP]
    if any(c > 1 for c in Counter(toks).values()):
        print(f"  greedy : {cap}")
        print(f"  beam {big}: {per_beam_caps[big][iid]}\n")
        shown += 1
        if shown >= 5:
            break
if shown == 0:
    print("  (none found - greedy produced no repeated content words on this set)")

print("saved:", rep_csv)
print("saved:", rep_png)

## 10c. Lowest greedy-BLEU images: greedy vs beam, side by side

Rank the eval images by their **greedy (beam 1) per-image BLEU-4**, take the worst `LOW_K`, and show each image with its **greedy** caption vs its **beam** caption (and a human reference). This is the qualitative view of *where greedy fails and whether beam fixes it* — useful for eyeballing repetition loops being broken. Reuses the captions already in `per_beam_caps`, so no re-decoding.

In [ ]:
smoothing = SmoothingFunction().method1

# Beam size to compare greedy against: best non-greedy by corpus BLEU, else widest.
_nz = res_df[res_df["beam_size"] != 1]
VS_BEAM = int(_nz.loc[_nz["corpus_bleu4"].idxmax(), "beam_size"]) if len(_nz) else BEAM_SIZES[-1]
LOW_K = 8

def img_bleu(caption, iid):
    hyp = word_tokenize(caption.lower())
    refs = [word_tokenize(c.lower()) for c in img_id_to_captions[iid]]
    return sentence_bleu(refs, hyp, weights=(0.25, 0.25, 0.25, 0.25),
                         smoothing_function=smoothing) if hyp else 0.0

rows = []
for iid in per_beam_caps[1]:
    g, b = per_beam_caps[1][iid], per_beam_caps[VS_BEAM][iid]
    rows.append({"image_id": iid, "greedy": g, "beam": b,
                 "greedy_bleu": img_bleu(g, iid), "beam_bleu": img_bleu(b, iid)})
rows.sort(key=lambda r: r["greedy_bleu"])
worst = rows[:LOW_K]

print(f"Lowest {LOW_K} greedy-BLEU images  (greedy vs beam {VS_BEAM}):\n")
for r in worst:
    print(f"  greedy  (BLEU {r['greedy_bleu']:.3f}): {r['greedy']}")
    print(f"  beam {VS_BEAM} (BLEU {r['beam_bleu']:.3f}): {r['beam']}")
    print(f"  ref: {img_id_to_captions[r['image_id']][0]}\n")

cols = min(4, LOW_K)
rows_n = math.ceil(LOW_K / cols)
fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 4.4, rows_n * 5.0))
axes = np.array(axes).reshape(-1)
for ax in axes:
    ax.axis("off")
for ax, r in zip(axes, worst):
    img = Image.open(os.path.join(img_dir, img_id_to_filename[r["image_id"]])).convert("RGB")
    ax.imshow(img)
    g = textwrap.fill(f"greedy ({r['greedy_bleu']:.2f}): {r['greedy']}", 40)
    b = textwrap.fill(f"beam{VS_BEAM} ({r['beam_bleu']:.2f}): {r['beam']}", 40)
    ax.set_title(g + "\n" + b, fontsize=8)
fig.suptitle(f"Lowest greedy-BLEU: greedy vs beam {VS_BEAM}  -  {MODEL_NAME}", fontsize=14, y=1.0)
fig.tight_layout()
low_png = os.path.join(OUTPUT_DIR, f"low_greedy_vs_beam_{MODEL_NAME}.png")
fig.savefig(low_png, dpi=150, bbox_inches="tight")
plt.show()
print("saved:", low_png)

## 11. Save outputs to Drive (optional)

In [ ]:
BEAM_DRIVE_DIR = "/content/drive/MyDrive/image_captioning_beam"
out_files = [res_csv, beam_png, ex_png]
out_files += [f for f in (globals().get('rep_csv'), globals().get('rep_png'), globals().get('low_png')) if f]
if ON_COLAB:
    os.makedirs(BEAM_DRIVE_DIR, exist_ok=True)
    for f in out_files:
        if os.path.exists(f):
            shutil.copy2(f, os.path.join(BEAM_DRIVE_DIR, os.path.basename(f)))
    print("Copied beam-search outputs to Drive:", BEAM_DRIVE_DIR)
else:
    print("Not on Colab - outputs persist locally under:", os.path.abspath(OUTPUT_DIR))